### Insert DataCite awards (provenance='datacite', priority=1) into openalex_awards_raw

Reads from `openalex.awards.datacite_awards` (materialized by the CreateDataCiteAwards DLT pipeline) and refreshes the priority=1 slice of `openalex.awards.openalex_awards_raw`.

Run downstream of the CreateDataCiteAwards DLT pipeline (DataCite_Awards task in CreateFunderSourcedAwards job). The `is_usable_award_id` filter rejects records whose `funder_award_id` would be unhelpful (DataCite DOI-suffix-style IDs like `2b6r-gx60` pass).

In [ ]:
%sql
-- Remove previous data for this source before inserting fresh data
DELETE FROM openalex.awards.openalex_awards_raw
WHERE provenance = 'datacite' AND priority = 1;

-- Insert into openalex_awards_raw with priority (run after DLT pipeline completes)
INSERT INTO openalex.awards.openalex_awards_raw
SELECT
    id,
    display_name,
    description,
    funder_id,
    funder_award_id,
    amount,
    currency,
    funder,
    funding_type,
    funder_scheme,
    provenance,
    start_date,
    end_date,
    start_year,
    end_year,
    lead_investigator,
    co_lead_investigator,
    investigators,
    landing_page_url,
    doi,
    works_api_url,
    created_date,
    updated_date,
    1 as priority
FROM openalex.awards.datacite_awards
WHERE openalex.common.is_usable_award_id(funder_award_id);
